# 8. PyTorch Tutorial 08 - Logistic Regression

In [40]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 0) Prepare data

In [41]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

n_samples, n_features = X.shape

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

## scale : recommended for logistic regression

### Standard scaling is recommended for logistic regression for a few reasons:

1. It speeds up convergence - Since logistic regression uses gradient descent optimization, standardizing the features helps the algorithm converge faster. This is because it prevents one feature from dominating the other features due to its larger scale.

2. It helps address multicollinearity - When features have very different scales, it can lead to multicollinearity between features which can impact the model performance. Standard scaling helps make the features more comparable and reduces multicollinearity.

3. It makes coefficients comparable - Since the features are scaled to have mean 0 and standard deviation 1 after standard scaling, the coefficients in the logistic regression model represent the change in log odds per standard deviation increase in that feature. This makes the coefficients more interpretable and comparable.

4. It can improve numerical stability - Logistic regression uses the Sigmoid or logistic function which can have numerical stability issues when the input values are too large. Standard scaling the features to a similar range can improve the numerical stability of the model.

So in summary, standard scaling helps with faster convergence, dealing with multicollinearity, making coefficients comparable and improving numerical stability - all of which benefit logistic regression performance and interpretability. Hence it is commonly recommended as a preprocessing step before fitting logistic regression models.

### Difference between .fit_transform() and .transform()
Here's why we use fit_transform for the train data and just transform for the test data in sklearn preprocessing:

1. fit_transform learns the transformer from the train data: When we call fit_transform on the train data, the transformer (e.g. StandardScaler) learns the mean and standard deviation from that data. 

2. transform then applies that transformation: When we then call transform on new data (test data), it applies the same transformation it learned from the train data. This ensures the test data is also scaled in the same way.

3. We don't want to learn from test data: If we called fit_transform on the test data as well, it would learn the mean and std from that data and apply a different transformation. This would make the train and test data inconsistent.

4. We want consistent transforms between train and test: In order for our model to generalize properly to unseen test data, the transforms applied to both the train and test data need to be consistent. By calling fit_transform on the train data and then transform on the test data, we ensure this.

5. fit_transform only needs to be called once: We only need to call fit_transform when initially fitting the transformer on the train data. Once that's done, we just call transform on any new data we want to transform in the same way.

So in summary, by calling fit_transform on the train data, the transformer learns the appropriate transformation from that data. Then when we call transform on the test data, it applies that same learned transformation, ensuring consistency between the train and test data. This consistency is important for the model to generalize properly.

In [42]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Cast to torch tensors

In [43]:
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))
X_test

tensor([[-0.7085,  0.1531, -0.7017,  ..., -0.5107,  0.5043,  0.2811],
        [-0.9578, -2.2431, -0.9696,  ..., -1.0445, -1.3419, -0.3880],
        [-0.4850, -0.6859, -0.3837,  ...,  0.2937,  0.5457,  1.0246],
        ...,
        [-0.2501, -1.0694, -0.3051,  ..., -0.2239, -1.4462, -1.1865],
        [-0.2042,  0.0625, -0.2593,  ...,  0.0175, -0.0284, -0.5634],
        [ 3.8046,  1.6058,  3.9461,  ...,  2.2410, -0.4304, -0.5335]])

## Make y_train and y_test column vectors

In [44]:
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

## 1) Model

In [45]:
# Linear model f = wx + b , sigmoid at the end
class LogisticRegression(nn.Module):
    def __init__(self, n_input_features):
        super().__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

model = LogisticRegression(n_features)

## 2) Loss and optimizer

In [52]:
num_epochs = 1000
learning_rate = 0.01
criterion = nn.BCELoss() # Binary Cross Entropy Loss
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Print the weights and bias for the trained model
print(f"Weights before training : {model.linear.weight}")
print(f"Bias before training: {model.linear.bias}")

Weights before training : Parameter containing:
tensor([[-0.9650, -0.9177, -0.9531, -1.0991, -0.2566,  0.6661, -1.2608, -1.3441,
         -0.0330,  0.8639, -2.0791,  0.5096, -0.9396, -1.9304, -0.6429,  1.3705,
         -0.2355, -0.4209,  0.6368,  1.1860, -1.5166, -2.2157, -1.1262, -1.6632,
         -2.1170,  0.2101, -1.7794, -1.3611, -1.4961, -0.6470]],
       requires_grad=True)
Bias before training: Parameter containing:
tensor([0.6382], requires_grad=True)


## 3) Training loop

In [53]:
for epoch in range(num_epochs):
    # Forward pass and loss
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train) # y_pred has to come in first position and y_train in second

    # Backward pass and update
    loss.backward()
    optimizer.step()
    
    # evaluation meanwhile
    accuracy = y_pred.round().eq(y_train).sum() / float(y_train.shape[0])

    # zero grad before new step
    optimizer.zero_grad()

    if (epoch+1) % 100 == 0:
        print(f'epoch: {epoch+1}, loss = {loss.item():.7f} accuracy : {accuracy.item():.7f}')

epoch: 100, loss = 0.0221264 accuracy : 0.9956044
epoch: 200, loss = 0.0221209 accuracy : 0.9956044
epoch: 300, loss = 0.0221153 accuracy : 0.9956044
epoch: 400, loss = 0.0221098 accuracy : 0.9956044
epoch: 500, loss = 0.0221043 accuracy : 0.9956044
epoch: 600, loss = 0.0220988 accuracy : 0.9956044
epoch: 700, loss = 0.0220933 accuracy : 0.9956044
epoch: 800, loss = 0.0220878 accuracy : 0.9956044
epoch: 900, loss = 0.0220823 accuracy : 0.9956044
epoch: 1000, loss = 0.0220768 accuracy : 0.9956044


In [48]:
with torch.no_grad():
    y_predicted = model(X_test)
    y_predicted_class = y_predicted.round()
    #print(y_predicted_class.eq(y_test))
    print(y_predicted_class.shape[0])

114


## Evaluation

In [49]:
with torch.no_grad():
    y_predicted = model(X_test)
    y_predicted_class = y_predicted.round()
    # .eq() method compares y_predicted_class adn y_test and returns True or False (convertible in 1 or 0)
    acc = y_predicted_class.eq(y_test).sum() / float(y_test.shape[0])
    print(f'accuracy: {acc.item():.4f}')

accuracy: 0.9649


In [51]:
# Print the weights and bias for the trained model
print(f"Weights after training : {model.linear.weight}")
print(f"Bias after training: {model.linear.bias}")

Weights after training : Parameter containing:
tensor([[-0.9650, -0.9177, -0.9531, -1.0991, -0.2566,  0.6661, -1.2608, -1.3441,
         -0.0330,  0.8639, -2.0791,  0.5096, -0.9396, -1.9304, -0.6429,  1.3705,
         -0.2355, -0.4209,  0.6368,  1.1860, -1.5166, -2.2157, -1.1262, -1.6632,
         -2.1170,  0.2101, -1.7794, -1.3611, -1.4961, -0.6470]],
       requires_grad=True)
Bias after training: Parameter containing:
tensor([0.6382], requires_grad=True)
